# Notebook 02: Preprocessing and Skill Profile Construction

## Objective
Transform the raw synthetic dataset into a clean benchmarking artifact and construct unified candidate skill profiles for downstream ATS scoring, ESCO normalization, and benchmarking.

## Methodology
1. Load raw dataset and confirm schema integrity
2. Handle missing values in education columns
3. Normalize education labels (map LLM to postgraduate tier)
4. Parse skill fields into token lists and deduplicate across fields
5. Construct unified skill profiles per candidate
6. Validate all outputs and export processed artifacts

## Assumptions
- Education missingness is random and safe to impute with a sentinel value
- LLM in highest_education is a synthetic generator artifact, not a domain-specific credential
- resume_length_words carries no signal and is excluded from downstream use
- Skill token deduplication is case-insensitive

## Inputs
- parsed_resumes.csv

## Outputs
- processed_resumes.csv
- candidate_skill_profiles.csv
- cleaned_skill_vocabulary.csv

In [2]:
# importing libraries
import pandas as pd
import numpy as np

# loading raw dataset
df = pd.read_csv("../Dataset/parsed_resumes.csv")
# confirming shape and dtypes
print("Shape:", df.shape)
print()
print(df.dtypes)
print()

# confirming missing value counts for education columns
print("Missing values - education columns:")
print(df[["highest_education", "education_field"]].isnull().sum())
print()

# confirming unique values in highest_education before normalization
print("highest_education unique values:")
print(df["highest_education"].value_counts(dropna=False))

Shape: (5000, 34)

candidate_id                               object
resume_file_name                           object
candidate_name                             object
primary_domain                             object
primary_role                               object
years_experience                            int64
highest_education                          object
education_field                            object
institution_tier                           object
technical_skills_raw                       object
tools_platforms_raw                        object
soft_skills_raw                            object
experience_summary                         object
project_summary                            object
key_achievements                           object
management_experience_flag                  int64
people_management_flag                      int64
project_management_experience_flag          int64
agile_scrum_experience_flag                 int64
client_facing_experience_flag  

## Step 1: Missing Value Handling and Education Normalization

Missing values in `highest_education` and `education_field` are imputed with the sentinel value "Unknown".
This avoids dropping records and preserves thin domain populations (Management, Engineering).

`LLM` in `highest_education` is mapped to `Postgraduate` rather than `Masters`.
This creates a clean three-tier postgraduate bucket (Masters, MBA, Postgraduate) without conflating
a synthetic artifact label with the genuine Masters label.
The downstream ATS scoring logic must treat Masters, MBA, and Postgraduate equivalently
when assigning the postgraduate education tier score.

`resume_length_words` is flagged as unusable and excluded from processed outputs.

In [3]:
# copy to avoid modifying original
processed = df.copy()

# impute missing education values with sentinel
processed["highest_education"] = processed["highest_education"].fillna("Unknown")
processed["education_field"] = processed["education_field"].fillna("Unknown")

# map LLM to Postgraduate
processed["highest_education"] = processed["highest_education"].replace("LLM", "Postgraduate")

# validation
print("highest_education after normalization:")
print(processed["highest_education"].value_counts(dropna=False))
print()

print("Remaining nulls in education columns:")
print(processed[["highest_education", "education_field"]].isnull().sum())
print()

# flag resume_length_words as unusable
# it is retained in the dataframe for auditability but excluded from all downstream scoring
EXCLUDED_COLUMNS = ["resume_length_words"]
print("Columns flagged as unusable (retained but excluded from scoring):", EXCLUDED_COLUMNS)

highest_education after normalization:
highest_education
Postgraduate    1216
Masters         1202
MBA             1185
Bachelors       1127
Unknown          270
Name: count, dtype: int64

Remaining nulls in education columns:
highest_education    0
education_field      0
dtype: int64

Columns flagged as unusable (retained but excluded from scoring): ['resume_length_words']


## Step 2: Skill Field Parsing

`technical_skills_raw` and `tools_platforms_raw` are comma-separated strings.
Each field is parsed into a list of cleaned tokens per candidate.

Cleaning steps applied to each token:
- Strip leading and trailing whitespace
- Lowercase
- Drop empty strings

No stemming or lemmatization at this stage. That is deferred to the ESCO normalization notebook
where token forms need to match taxonomy entries deliberately.

In [4]:
# parsing a comma-separated skill string into a cleaned token list
def parse_skill_field(raw_value):
    if pd.isna(raw_value) or str(raw_value).strip() == "":
        return []
    tokens = [t.strip().lower() for t in str(raw_value).split(",")]
    return [t for t in tokens if t != ""]

# applying to both skill fields
processed["technical_skills_list"] = processed["technical_skills_raw"].apply(parse_skill_field)
processed["tools_platforms_list"] = processed["tools_platforms_raw"].apply(parse_skill_field)

# spot check: first 3 records
for i in range(3):
    print(f"Record {i}")
    print("  technical:", processed["technical_skills_list"].iloc[i])
    print("  tools:    ", processed["tools_platforms_list"].iloc[i])
    print()

# distribution check: how many tokens per field on average
tech_lengths = processed["technical_skills_list"].apply(len)
tools_lengths = processed["tools_platforms_list"].apply(len)

print("technical_skills_list token count - min/mean/max:",
      tech_lengths.min(), round(tech_lengths.mean(), 2), tech_lengths.max())
print("tools_platforms_list token count  - min/mean/max:",
      tools_lengths.min(), round(tools_lengths.mean(), 2), tools_lengths.max())

# confirm no empty skill profiles
print()
print("Records with empty technical_skills_list:", (tech_lengths == 0).sum())
print("Records with empty tools_platforms_list: ", (tools_lengths == 0).sum())

Record 0
  technical: ['machine learning', 'power bi', 'pandas', 'python']
  tools:     ['aws', 'servicenow']

Record 1
  technical: ['kubernetes', 'git', 'docker', 'python']
  tools:     ['aws', 'jira']

Record 2
  technical: ['due diligence', 'legal research', 'compliance', 'contract drafting']
  tools:     ['servicenow', 'aws']

technical_skills_list token count - min/mean/max: 4 4.0 4
tools_platforms_list token count  - min/mean/max: 2 2.0 2

Records with empty technical_skills_list: 0
Records with empty tools_platforms_list:  0


## Step 3: Skill Deduplication and Unified Profile Construction

Each candidate's technical skills and tools are merged into a single unified skill profile.
Deduplication is applied after merging to eliminate tokens that appear in both fields.

Deduplication is case-insensitive. Since both lists are already lowercased at parse time,
set-based deduplication is sufficient.

The unified profile is stored as a sorted list for deterministic output ordering.
Sorting has no downstream effect on matching or scoring but makes validation and diffing easier.

In [5]:
# merging and deduplicating skill tokens per candidate
def build_skill_profile(tech_list, tools_list):
    combined = set(tech_list + tools_list)
    return sorted(combined)

processed["skill_profile"] = processed.apply(
    lambda row: build_skill_profile(row["technical_skills_list"], row["tools_platforms_list"]),
    axis=1
)

# spot check: records where deduplication would have effect
# look for candidates where aws appears in both fields
overlap_mask = processed.apply(
    lambda row: len(set(row["technical_skills_list"]) & set(row["tools_platforms_list"])) > 0,
    axis=1
)
print("Records with overlap between technical and tools fields:", overlap_mask.sum())
print()

# show a sample overlap record
if overlap_mask.sum() > 0:
    sample = processed[overlap_mask].iloc[0]
    print("Sample overlap record:")
    print("  technical:", sample["technical_skills_list"])
    print("  tools:    ", sample["tools_platforms_list"])
    print("  unified:  ", sample["skill_profile"])
    print()

# profile length distribution
profile_lengths = processed["skill_profile"].apply(len)
print("skill_profile token count - min/mean/max:",
      profile_lengths.min(), round(profile_lengths.mean(), 2), profile_lengths.max())
print("Records with empty skill_profile:", (profile_lengths == 0).sum())

Records with overlap between technical and tools fields: 451

Sample overlap record:
  technical: ['java', 'kubernetes', 'git', 'aws']
  tools:     ['aws', 'servicenow']
  unified:   ['aws', 'git', 'java', 'kubernetes', 'servicenow']

skill_profile token count - min/mean/max: 5 5.91 6
Records with empty skill_profile: 0


## Step 4: Vocabulary Extraction and Artifact Export

The global skill vocabulary is the set of all unique tokens across all candidate skill profiles.
This vocabulary is exported as a reference artifact for the ESCO normalization notebook.

Three artifacts are exported:
- `processed_resumes.csv`: cleaned dataset with all original columns plus normalized fields
- `candidate_skill_profiles.csv`: candidate_id mapped to skill profile as a pipe-delimited string
- `cleaned_skill_vocabulary.csv`: all unique skill tokens with frequency counts

In [6]:
import os

output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

# building vocabulary with frequency counts
from collections import Counter

all_tokens = [token for profile in processed["skill_profile"] for token in profile]
vocab_counter = Counter(all_tokens)
vocab_df = pd.DataFrame(vocab_counter.items(), columns=["skill_token", "frequency"])
vocab_df["requires_review"] = vocab_df["frequency"] == 1
vocab_df = vocab_df.sort_values("frequency", ascending=False).reset_index(drop=True)

print("Total unique skill tokens:", len(vocab_df))
print()
print("Top 20 tokens by frequency:")
print(vocab_df.head(20).to_string(index=False))
print()
print("Bottom 10 tokens by frequency:")
print(vocab_df.tail(10).to_string(index=False))
print()
print("Tokens flagged for review:", vocab_df["requires_review"].sum())

Total unique skill tokens: 217

Top 20 tokens by frequency:
     skill_token  frequency  requires_review
             aws       2583            False
             sql       1923            False
          python       1796            False
             gcp       1712            False
      servicenow       1697            False
           azure       1695            False
      confluence       1628            False
            jira       1622            False
             git       1382            False
          docker       1298            False
           linux       1293            False
      kubernetes       1271            False
            java       1251            False
             nlp        561            False
      tensorflow        554            False
        power bi        546            False
          pandas        526            False
machine learning        522            False
            hrms        383            False
     recruitment        381            F

In [7]:
# exporting vocabulary
vocab_df.to_csv("../outputs/cleaned_skill_vocabulary.csv", index=False)
print("cleaned_skill_vocabulary.csv exported:", vocab_df.shape)
print()

# exporting candidate skill profiles
profiles_df = processed[["candidate_id"]].copy()
profiles_df["skill_profile"] = processed["skill_profile"].apply(lambda x: "|".join(x))
profiles_df["skill_count"] = processed["skill_profile"].apply(len)
profiles_df.to_csv("../outputs/candidate_skill_profiles.csv", index=False)
print("candidate_skill_profiles.csv exported:", profiles_df.shape)
print()

# exporting processed resumes
# dropping intermediate list columns before export
export_cols = [c for c in processed.columns if c not in ["technical_skills_list", "tools_platforms_list"]]
processed_export = processed[export_cols].copy()
processed_export["skill_profile"] = processed["skill_profile"].apply(lambda x: "|".join(x))
processed_export.to_csv("../outputs/processed_resumes.csv", index=False)
print("processed_resumes.csv exported:", processed_export.shape)

cleaned_skill_vocabulary.csv exported: (217, 3)

candidate_skill_profiles.csv exported: (5000, 3)

processed_resumes.csv exported: (5000, 35)


In [10]:
# loading exported artifacts and validating success criteria
processed_check = pd.read_csv("../outputs/processed_resumes.csv")
profiles_check = pd.read_csv("../outputs/candidate_skill_profiles.csv")
vocab_check = pd.read_csv("../outputs/cleaned_skill_vocabulary.csv")

print(" processed_resumes.csv ")
print("Shape:", processed_check.shape)
print("Nulls in highest_education:", processed_check["highest_education"].isnull().sum())
print("Nulls in education_field:", processed_check["education_field"].isnull().sum())
print("LLM remaining:", (processed_check["highest_education"] == "LLM").sum())
print("Education values:", processed_check["highest_education"].value_counts().to_dict())
print()

print("candidate_skill_profiles.csv ")
print("Shape:", profiles_check.shape)
print("Empty profiles:", (profiles_check["skill_profile"] == "").sum())
print("Null profiles:", profiles_check["skill_profile"].isnull().sum())
print("skill_count min/mean/max:",
      profiles_check["skill_count"].min(),
      round(profiles_check["skill_count"].mean(), 2),
      profiles_check["skill_count"].max())

# confirm aws is not double-counted
sample_aws = profiles_check[profiles_check["skill_profile"].str.contains("aws")].head(3)
for _, row in sample_aws.iterrows():
    tokens = row["skill_profile"].split("|")
    aws_count = tokens.count("aws")
    print(f"  candidate {row['candidate_id']}: aws appears {aws_count}x in profile")
print()

print(" cleaned_skill_vocabulary.csv ")
print("Shape:", vocab_check.shape)
print("Tokens flagged for review:", vocab_check["requires_review"].sum())

 processed_resumes.csv 
Shape: (5000, 35)
Nulls in highest_education: 0
Nulls in education_field: 0
LLM remaining: 0
Education values: {'Postgraduate': 1216, 'Masters': 1202, 'MBA': 1185, 'Bachelors': 1127, 'Unknown': 270}

candidate_skill_profiles.csv 
Shape: (5000, 3)
Empty profiles: 0
Null profiles: 0
skill_count min/mean/max: 5 5.91 6
  candidate CND_100001: aws appears 1x in profile
  candidate CND_100002: aws appears 1x in profile
  candidate CND_100003: aws appears 1x in profile

 cleaned_skill_vocabulary.csv 
Shape: (217, 3)
Tokens flagged for review: 44
